# **Redefining classes from training notebook**
You can look at my trainign notebook for a better understanding of the code
[Training Notebook](https://www.kaggle.com/code/marcusewang/baseline-efficient-net/)


In [1]:
#Marcus Wang
import re
import random
import matplotlib.pyplot as plt
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import torch
import torchaudio
import torchaudio.transforms as T
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import timm
from torchvision.transforms import v2
from torch.utils.data import Dataset, DataLoader
PATH = '/kaggle/input/competitions/birdclef-2026/test_soundscapes/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

For inference notebooks, whenever you try to print the length, or index into testsoundscapes, it will always give you 0, because it is done again in a seperate server where you don't have access to any outputs, and only the lb score.

In [2]:
#taxonomy
taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')

#test soundscapes
testSounds = [os.path.join(PATH, afile) for afile in sorted(os.listdir(PATH)) if afile.endswith('.ogg')]
print(len(testSounds))

#train soundscapes
trainSounds = pd.DataFrame(columns = ["filename"])
for dirname, _, filenames in os.walk('/kaggle/input/competitions/birdclef-2026/train_soundscapes/'):
    for filename in filenames:
        trainSounds.loc[len(trainSounds)] = str(os.path.join(dirname, filename))

#sample_submission example
exsub = pd.read_csv('/kaggle/input/competitions/birdclef-2026/sample_submission.csv')
print("filenames")

0
filenames


In [3]:
print("example submission")
exsub.head()

example submission


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


In [4]:
# - Variables - - - - - - - - - - - - - - - - - - #
nclasses = 234 #there are 234 class choices for submission
framesForFive = 313 #number of frames for spectrogram for 5 seconds

nblocks = 5
# - - - - - - - - - - - - - - - - - - - - - - - - #

# - taxonomy id to id number and vice versa - - - #
taxIds = taxonomy["primary_label"].tolist()
taxToNum = {s:i for i,s in enumerate(taxIds)}
numToTax = {i:s for i,s in enumerate(taxIds)}
# - - - - - - - - - - - - - - - - - - - - - - - - #

In [5]:
class BatchTransform():
    def __init__(self,):
        
        self.melTransform = T.MelSpectrogram(
            sample_rate=32000,
            n_fft=1024,
            win_length=None,
            hop_length=512,
            n_mels=256,
            f_min=250,
            mel_scale  = 'htk',
        )
        self.dbTransform = T.AmplitudeToDB()
        self.DUR = 5 * 32000

    def get(self, filename):
        spectrograms = []
        waveform, sampleRate = torchaudio.load(filename)
        
        for i in range(12):
            piece = waveform[:, i * self.DUR : (i+1) * self.DUR]
            melSpectrogram = self.dbTransform(self.melTransform(piece))
            spectrograms.append(melSpectrogram)
            
        return torch.stack(spectrograms)
testDataset = BatchTransform()

/usr/local/lib/python3.12/dist-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (256) may be set too high. Or, the value for `n_freqs` (513) may be set too low.
  warnings.warn(


In [6]:
#Building the Gem Architecture which is not avaliable in pytorch
class GeM(nn.Module):
    def __init__(self, p=2, eps=1e-6):
        super(GeM,self).__init__()
        #p is a learnable parameter
        self.p = nn.Parameter(torch.ones(1)*p)
        self.eps = eps

    def forward(self, x):
        return self.gem(x, p=self.p)
        
    def gem(self, x, p=2):
        xpk = x.clamp(min = self.eps).pow(p)
        #                            H            W
        avgPool = F.avg_pool2d(xpk, (x.size(-2), x.size(-1)))
        return avgPool.pow(1./p)

In [7]:
class GeMModel(nn.Module):
    def __init__(self, model_name='hgnetv2_b0.ssld_stage2_ft_in1k', pretrained=True):
        super().__init__()
        #loads the efficient net architecture and weights
        self.backbone = timm.create_model(
            model_name, 
            pretrained=pretrained, 
            in_chans=1, 
            num_classes=0,       #removes the final Linear layer
            global_pool=''       #removes the GAP layer
        )
        
        self.pool = GeM()
        in_features = self.backbone.num_features*2
        self.head = nn.Sequential(nn.Linear(in_features, in_features//2),
                                  nn.ReLU(inplace = True), nn.Dropout(p=0.2),
                                  nn.Linear(in_features//2,nclasses)
                                 )

    def forward(self, x):
        
        x = self.backbone(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        logits = self.head(x)
        
        return logits

In [8]:
finalModel = GeMModel(pretrained = False)
finalModel.to(device)
lossi = []

# **Load Trained Model and Predict**

In [9]:
#load the model which I trained externally
state_dict = torch.load("/kaggle/input/models/marcusewang/gem-hg-net-v2-b0/pytorch/default/1/final_HGNetV2_b0.pth",map_location=device)
finalModel.load_state_dict(state_dict['model_state_dict'])

<All keys matched successfully>

For prediction, you cannot use anything which is based off of the length of the data, because Kaggle hides the data making something like "testSounds[i]" not possible. To solve this you have to use "in" which does through every data in the dataset without kaggle freaking out and hiding the data.

In [10]:
predictions = pd.DataFrame(columns=list(exsub.columns))
finalModel.eval()
with torch.no_grad():
    for sound in testSounds: 
        batch = testDataset.get(sound)
        batch = batch.to(device)
        
        logits = finalModel(batch)
        preds = torch.sigmoid(logits)
        
        raw_filename = os.path.basename(sound).replace('.ogg', '')
        for j, pred in enumerate(preds):
            # Formats as: "BC2026_Test_0001_5"
            row_id = f"{raw_filename}_{j * 5 + 5}" 
            
            predictions.loc[len(predictions)] = [row_id] + pred.cpu().tolist()

predictions.head()

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1


In [11]:
predictions.to_csv("submission.csv", index=False)